In [1]:
pip install pandas openpyxl openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import pandas as pd
import ast
from getpass import getpass
from openai import OpenAI

# =========================
# PATHS
# =========================
INPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/sentence_similarity_results/k_32"
OUTPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/SV-SSim"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# API
# =========================
api_key = getpass("Enter OpenAI API Key: ")
client = OpenAI(api_key=api_key)

# =========================
# FIXED PARSER (IMPORTANT)
# =========================
def parse_names(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    x = str(x).strip()

    # python list string
    if x.startswith("["):
        try:
            return ast.literal_eval(x)
        except:
            return []

    # semicolon separated (MOST IMPORTANT FIX)
    if ";" in x:
        return [i.strip() for i in x.split(";") if i.strip()]

    # single value
    if x == "" or x.lower() == "none":
        return []

    return [x]

# =========================
# SELF VERIFICATION
# =========================
def verify_name(sentence, word):
    prompt = f"""
Sentence: {sentence}

Word: {word}

Is "{word}" referring to a person name in this sentence?
Answer only Yes or No.
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            temperature=0,
            messages=[{"role": "user", "content": prompt}]
        )
        ans = response.choices[0].message.content.strip().lower()
        return "yes" in ans
    except Exception as e:
        print(f"API Error: {e}")
        return False

# =========================
# METRICS
# =========================
def compute_metrics(preds, golds):
    TP = FP = FN = 0

    for p, g in zip(preds, golds):
        p_set = set(p)
        g_set = set(g)

        TP += len(p_set & g_set)
        FP += len(p_set - g_set)
        FN += len(g_set - p_set)

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall    = TP / (TP + FN) if TP + FN > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return precision, recall, f1

# =========================
# MAIN LOOP
# =========================
all_f1 = []

for fold in range(1, 6):
    print(f"\n===== Processing Fold {fold} =====")

    file_path = os.path.join(INPUT_DIR, f"predictions_fold_{fold}.xlsx")
    df = pd.read_excel(file_path)

    print("Columns:", df.columns.tolist())

    sv_results = []

    for idx, row in df.iterrows():
        sentence = row["sentence"]
        pred_names = parse_names(row["pred_names_parsed"])

        # DEBUG (first few rows only)
        if idx < 3:
            print("\n--- DEBUG ---")
            print("Sentence:", sentence[:80])
            print("Pred:", pred_names)

        verified = []

        for name in pred_names:
            if verify_name(sentence, name):
                verified.append(name)

        if idx < 3:
            print("Verified:", verified)

        sv_results.append(verified)

    df["SV_pred_names"] = sv_results

    # =========================
    # METRICS
    # =========================
    gold = df["gold_names_parsed"].apply(parse_names)
    pred_after = df["SV_pred_names"]

    P, R, F1 = compute_metrics(pred_after, gold)

    print(f"\nFold {fold} Results:")
    print(f"Precision: {P:.4f}")
    print(f"Recall   : {R:.4f}")
    print(f"F1       : {F1:.4f}")

    all_f1.append(F1)

    # =========================
    # SAVE FILE
    # =========================
    output_path = os.path.join(OUTPUT_DIR, f"SV-SSim_fold_{fold}.xlsx")
    df.to_excel(output_path, index=False)

# =========================
# CROSS VALIDATION
# =========================
cv_f1 = sum(all_f1) / len(all_f1)

print("\n==============================")
print(f"Cross Validation F1: {cv_f1:.4f}")
print("==============================")


===== Processing Fold 1 =====
Columns: ['sentence', 'gold_name_raw', 'gold_names_parsed', 'model_output', 'pred_names_parsed', 'row_tp', 'row_fp', 'row_fn', 'retrieved_demo_1', 'retrieval_score_1', 'retrieved_demo_2', 'retrieval_score_2', 'retrieved_demo_3', 'retrieval_score_3', 'retrieved_demo_4', 'retrieval_score_4', 'retrieved_demo_5', 'retrieval_score_5', 'retrieved_demo_6', 'retrieval_score_6', 'retrieved_demo_7', 'retrieval_score_7', 'retrieved_demo_8', 'retrieval_score_8']

--- DEBUG ---
Sentence: Him þa Andreas ondsware agef: "Hwæt frinest ðu me, frea leofesta, wordum wrætlic
Pred: ['Andreas']
Verified: ['Andreas']

--- DEBUG ---
Sentence: Gewat þa Matheus menigo lædan on gehyld godes, swa him se halga bebead.
Pred: ['Matheus']
Verified: ['Matheus']

--- DEBUG ---
Sentence: Him þa Andreas agef ondsware: "Hwæt, ðu þristlice þeode lærest, bældest to beado
Pred: ['Andreas']
Verified: ['Andreas']

Fold 1 Results:
Precision: 0.9846
Recall   : 0.9412
F1       : 0.9624

===== Process

In [4]:
import os
import pandas as pd
import ast

# ===== PATH =====
INPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/SV-SSim"
OUTPUT_FILE = os.path.join(INPUT_DIR, "SV-SSim_metrics_summary.xlsx")

# ===== parser =====
def parse_names(x):
    if pd.isna(x):
        return []
    x = str(x).strip()

    if x.startswith("["):
        try:
            return ast.literal_eval(x)
        except:
            return []

    if ";" in x:
        return [i.strip() for i in x.split(";") if i.strip()]

    if x == "" or x.lower() == "none":
        return []

    return [x]

# ===== metric =====
def compute_metrics(preds, golds):
    TP = FP = FN = 0

    for p, g in zip(preds, golds):
        p, g = set(p), set(g)
        TP += len(p & g)
        FP += len(p - g)
        FN += len(g - p)

    P = TP/(TP+FP) if TP+FP else 0
    R = TP/(TP+FN) if TP+FN else 0
    F1 = 2*P*R/(P+R) if P+R else 0

    return P, R, F1

# ===== process folds =====
results = []

for fold in range(1, 6):
    file_path = os.path.join(INPUT_DIR, f"SV-SSim_fold_{fold}.xlsx")
    df = pd.read_excel(file_path)

    gold = df["gold_names_parsed"].apply(parse_names)
    pred = df["SV_pred_names"].apply(parse_names)

    P, R, F1 = compute_metrics(pred, gold)

    results.append({
        "fold": fold,
        "precision": P,
        "recall": R,
        "f1_score": F1
    })

# ===== create summary =====
summary_df = pd.DataFrame(results)

# average row
avg_f1 = summary_df["f1_score"].mean()
avg_precision = summary_df["precision"].mean()
avg_recall = summary_df["recall"].mean()

summary_df.loc["avg"] = ["-", avg_precision, avg_recall, avg_f1]

# ===== save =====
summary_df.to_excel(OUTPUT_FILE, index=False)

print("Saved:", OUTPUT_FILE)
print("\nSummary:\n", summary_df)

Saved: /Users/shinekhantaung/Desktop/GPT-NER/Sentence-Retrieval/SV-SSim/SV-SSim_metrics_summary.xlsx

Summary:
     fold  precision    recall  f1_score
0      1   0.984615  0.941176  0.962406
1      2   0.954357  0.927419  0.940695
2      3   0.972603  0.922078  0.946667
3      4   0.974576  0.927419  0.950413
4      5   0.967213  0.944000  0.955466
avg    -   0.970673  0.932419  0.951129
